# Módulo I8 · Del notebook a producción — Práctica guiada

**Caso:** TiendaSur quiere saber qué clientes tienen riesgo de abandonar para lanzarles una campaña de retención. Un compañero tiene un notebook con un modelo que «funciona». Tu trabajo es convertirlo en algo que **otra persona y otra máquina** puedan usar: un paquete con tests, una API, una imagen Docker, CI, seguimiento de experimentos y monitorización.

Todo se escribe en la carpeta `proyecto-churn/` junto a este notebook. Al final tendrás un repositorio completo que puedes copiar, subir a GitHub y enseñar en una entrevista.

> **Requisitos:** `pip install -r requirements.txt` del curso (incluye fastapi, pytest, ruff y mlflow). Docker es opcional.


In [ ]:
import importlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Todo el proyecto se escribe en esta carpeta (está en .gitignore del curso).
BASE = Path("proyecto-churn").resolve()
if BASE.exists():
    shutil.rmtree(BASE)          # empezamos siempre de cero: reproducibilidad
PAQUETE = BASE / "src" / "tiendasur_churn"
PAQUETE.mkdir(parents=True)
(BASE / "tests").mkdir()

def escribir(ruta_relativa: str, contenido: str) -> None:
    """Escribe un archivo del proyecto y muestra su ruta."""
    ruta = BASE / ruta_relativa
    ruta.parent.mkdir(parents=True, exist_ok=True)
    ruta.write_text(contenido.lstrip("\n"))
    print(f"escrito: {ruta.relative_to(BASE.parent)}  ({len(contenido.splitlines())} líneas)")

def ejecutar(*args: str) -> int:
    """Ejecuta un comando dentro del proyecto (como harías en la terminal) y muestra su salida."""
    entorno = {**os.environ, "PYTHONPATH": str(BASE / "src")}   # como `PYTHONPATH=src` en la terminal
    r = subprocess.run(list(args), cwd=BASE, capture_output=True, text=True, env=entorno)
    print((r.stdout + r.stderr).strip()[-3000:])
    return r.returncode

sys.path.insert(0, str(BASE / "src"))
print("Proyecto en:", BASE)

## 8.1 · Del notebook al paquete

Un notebook es ideal para explorar, pero no se puede importar, testear ni desplegar con fiabilidad. El primer paso es **mover el código a funciones dentro de un paquete** (`src/tiendasur_churn/`). Regla práctica: si una celda se va a ejecutar más de una vez, es una función; si una función se usa en producción, vive en un `.py` con test.

`pyproject.toml` declara el proyecto y la configuración de las herramientas (pytest, ruff) en un único sitio.

In [ ]:
escribir("pyproject.toml", """
[project]
name = "tiendasur-churn"
version = "0.1.0"
requires-python = ">=3.10"

[tool.pytest.ini_options]
pythonpath = ["src"]
testpaths = ["tests"]

[tool.ruff]
line-length = 100
src = ["src"]

[tool.ruff.lint]
select = ["E", "F", "I", "B"]   # errores, imports sin usar, orden de imports, bugs habituales
""")
escribir("src/tiendasur_churn/__init__.py", '"""Modelo de abandono de clientes de TiendaSur."""\n')

### Datos con contrato

`validar_esquema` falla **pronto y con un mensaje claro** si los datos no son los esperados. Es mucho más barato que descubrir en producción que una columna cambió de nombre.

In [ ]:
escribir("src/tiendasur_churn/datos.py", """
\"\"\"Carga (aquí: simulación) y validación de los datos de clientes.\"\"\"
import numpy as np
import pandas as pd

NUMERICAS = ["antiguedad_meses", "pedidos_ultimo_ano", "ticket_medio",
             "dias_desde_ultimo_pedido", "incidencias_soporte"]
CATEGORICAS = ["plan", "canal"]
OBJETIVO = "abandona"
PLANES = ["basico", "plus", "premium"]
CANALES = ["web", "app", "tienda"]


def generar_datos(n: int = 5000, semilla: int = 42) -> pd.DataFrame:
    \"\"\"Simula clientes con una relación realista entre comportamiento y abandono.\"\"\"
    rng = np.random.default_rng(semilla)
    df = pd.DataFrame({
        "cliente_id": np.arange(1, n + 1),
        "antiguedad_meses": rng.integers(0, 73, n),
        "pedidos_ultimo_ano": rng.poisson(6, n),
        "ticket_medio": np.round(rng.lognormal(3.4, 0.5, n), 2),
        "dias_desde_ultimo_pedido": rng.integers(0, 366, n),
        "incidencias_soporte": rng.poisson(0.6, n),
        "plan": rng.choice(PLANES, n, p=[0.6, 0.3, 0.1]),
        "canal": rng.choice(CANALES, n, p=[0.5, 0.35, 0.15]),
    })
    efecto_plan = df["plan"].map({"basico": 0.4, "plus": 0.0, "premium": -0.6})
    logit = (-1.8 + 0.012 * df["dias_desde_ultimo_pedido"] - 0.03 * df["antiguedad_meses"]
             - 0.15 * df["pedidos_ultimo_ano"] + 0.7 * df["incidencias_soporte"] + efecto_plan)
    df[OBJETIVO] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    return df


def validar_esquema(df: pd.DataFrame) -> pd.DataFrame:
    \"\"\"Falla pronto y con un mensaje claro si los datos no son los que el modelo espera.\"\"\"
    faltan = set(NUMERICAS + CATEGORICAS) - set(df.columns)
    if faltan:
        raise ValueError(f"faltan columnas: {sorted(faltan)}")
    if (df[NUMERICAS] < 0).any().any():
        raise ValueError("hay valores numéricos negativos")
    desconocidos = set(df["plan"]) - set(PLANES)
    if desconocidos:
        raise ValueError(f"planes desconocidos: {sorted(desconocidos)}")
    return df
""")

import tiendasur_churn.datos as datos
importlib.reload(datos)
df = datos.validar_esquema(datos.generar_datos())
print(df.shape, "| tasa de abandono:", f"{df['abandona'].mean():.1%}")
df.head()

## 8.2 · Pipelines reproducibles y sin fugas

Todo el preprocesado (escalado, one-hot) va **dentro** de un `Pipeline` de scikit-learn. Así:
- se ajusta solo con los datos de entrenamiento (sin fuga de información del test);
- el mismo objeto que entrenas es el que despliegas: no hay que reescribir el preprocesado en la API.

Junto al modelo guardamos una **ficha** (`ficha.json`): métricas, huella de los datos y versiones de las librerías. Sin ella, dentro de seis meses nadie sabrá qué hay en producción.

In [ ]:
escribir("src/tiendasur_churn/modelo.py", """
\"\"\"Pipeline de entrenamiento: todo el preprocesado vive DENTRO del pipeline.\"\"\"
import hashlib
import json
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from tiendasur_churn.datos import CATEGORICAS, NUMERICAS, OBJETIVO

# Valores por defecto prudentes: árboles poco profundos y aprendizaje lento sobreajustan menos.
PARAMS_DEFECTO = {"max_depth": 3, "learning_rate": 0.05, "max_iter": 300}


def construir_pipeline(**params) -> Pipeline:
    preprocesado = ColumnTransformer([
        ("num", StandardScaler(), NUMERICAS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
    ])
    clasificador = HistGradientBoostingClassifier(random_state=0, **{**PARAMS_DEFECTO, **params})
    return Pipeline([("prep", preprocesado), ("modelo", clasificador)])


def entrenar(df: pd.DataFrame, semilla: int = 0, **params):
    \"\"\"Devuelve (pipeline entrenado, métricas en test). El test nunca se toca al ajustar.\"\"\"
    X, y = df[NUMERICAS + CATEGORICAS], df[OBJETIVO]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y,
                                              random_state=semilla)
    pipe = construir_pipeline(**params).fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])
    return pipe, {"auc_test": round(float(auc), 4), "n_train": len(X_tr), "n_test": len(X_te)}


def huella_datos(df: pd.DataFrame) -> str:
    \"\"\"Hash del dataset: si cambian los datos, cambia la huella (trazabilidad).\"\"\"
    return hashlib.sha256(pd.util.hash_pandas_object(df, index=False).values).hexdigest()[:12]


def guardar(pipe: Pipeline, metricas: dict, df: pd.DataFrame, carpeta: Path) -> Path:
    \"\"\"Guarda el modelo y, a su lado, la ficha con todo lo necesario para reproducirlo.\"\"\"
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    joblib.dump(pipe, carpeta / "modelo.joblib")
    ficha = {
        "version_modelo": datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S"),
        "metricas": metricas,
        "huella_datos": huella_datos(df),
        "columnas": NUMERICAS + CATEGORICAS,
        "versiones": {p: version(p) for p in ["scikit-learn", "pandas", "numpy"]},
    }
    (carpeta / "ficha.json").write_text(json.dumps(ficha, indent=2, ensure_ascii=False))
    return carpeta / "modelo.joblib"
""")

import tiendasur_churn.modelo as modelo
importlib.reload(modelo)
pipe, metricas = modelo.entrenar(df)
print("métricas:", metricas)
ruta_modelo = modelo.guardar(pipe, metricas, df, BASE / "artefactos")
print(json.dumps(json.loads((BASE / "artefactos" / "ficha.json").read_text()), indent=2, ensure_ascii=False))

In [ ]:
# Anti-patrón: escalar ANTES de separar train/test. El scaler "ve" el test (fuga de información).
# Con StandardScaler el efecto es pequeño; con imputación por la media, target encoding o selección
# de variables puede inflar la métrica mucho. El Pipeline lo impide por construcción:
# el preprocesado se ajusta en cada fit() solo con los datos de entrenamiento.
from sklearn.preprocessing import StandardScaler
X_mal = StandardScaler().fit_transform(df[datos.NUMERICAS])   # ← fit con TODO el dataset
print("media del scaler calculada con", len(X_mal), "filas (incluye las de test): eso es la fuga")

### Del notebook a un comando

En producción nadie abre un notebook para reentrenar: se ejecuta un comando. `entrenar.py` es el punto de entrada reproducible que usará también el CI.

In [ ]:
escribir("src/tiendasur_churn/entrenar.py", """
\"\"\"Punto de entrada reproducible:  python -m tiendasur_churn.entrenar --salida artefactos\"\"\"
import argparse

from tiendasur_churn.datos import generar_datos, validar_esquema
from tiendasur_churn.modelo import entrenar, guardar


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--salida", default="artefactos")
    parser.add_argument("--semilla", type=int, default=42)
    args = parser.parse_args()
    df = validar_esquema(generar_datos(semilla=args.semilla))
    pipe, metricas = entrenar(df)
    print("guardado en", guardar(pipe, metricas, df, args.salida), metricas)


if __name__ == "__main__":
    main()
""")
ejecutar(sys.executable, "-m", "tiendasur_churn.entrenar", "--salida", "artefactos")

## 8.3 · Servir el modelo con FastAPI

La API tiene tres ideas clave:
1. **Contrato de entrada con Pydantic** (`Cliente`): tipos, rangos y valores permitidos. Lo que no lo cumpla recibe un `422` antes de tocar el modelo.
2. **El modelo se carga una vez** al arrancar (`lifespan`), no en cada petición.
3. **`/health`** para que el orquestador (Docker, Kubernetes, un balanceador) sepa si el servicio está vivo, y **la versión del modelo en cada respuesta** para poder rastrear qué modelo tomó cada decisión.

In [ ]:
escribir("src/tiendasur_churn/api.py", """
\"\"\"API de predicción. Arranque local:  uvicorn tiendasur_churn.api:app --reload\"\"\"
import json
import logging
import os
import time
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Literal

import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

UMBRAL = float(os.environ.get("UMBRAL_ABANDONO", "0.5"))
log = logging.getLogger("tiendasur_churn")
estado: dict = {}


class Cliente(BaseModel):
    \"\"\"Contrato de entrada: lo que no cumpla esto recibe un 422 antes de tocar el modelo.\"\"\"
    antiguedad_meses: int = Field(ge=0, le=600)
    pedidos_ultimo_ano: int = Field(ge=0)
    ticket_medio: float = Field(gt=0)
    dias_desde_ultimo_pedido: int = Field(ge=0)
    incidencias_soporte: int = Field(ge=0)
    plan: Literal["basico", "plus", "premium"]
    canal: Literal["web", "app", "tienda"]


class Prediccion(BaseModel):
    probabilidad_abandono: float
    accion_retencion: bool
    version_modelo: str


@asynccontextmanager
async def ciclo_de_vida(app: FastAPI):
    # El modelo se carga UNA vez al arrancar, no en cada petición.
    ruta = Path(os.environ.get("MODELO_PATH", "artefactos/modelo.joblib"))
    estado["modelo"] = joblib.load(ruta)
    estado["ficha"] = json.loads((ruta.parent / "ficha.json").read_text())
    yield
    estado.clear()


app = FastAPI(title="TiendaSur · abandono de clientes", lifespan=ciclo_de_vida)


@app.get("/health")
def health() -> dict:
    return {"estado": "ok", "version_modelo": estado["ficha"]["version_modelo"]}


@app.post("/predict", response_model=Prediccion)
def predict(cliente: Cliente) -> Prediccion:
    inicio = time.perf_counter()
    X = pd.DataFrame([cliente.model_dump()])
    p = float(estado["modelo"].predict_proba(X)[0, 1])
    version_modelo = estado["ficha"]["version_modelo"]
    # Log estructurado (JSON): sin datos personales, con lo necesario para monitorizar.
    ms = round(1000 * (time.perf_counter() - inicio), 1)
    log.info(json.dumps({"evento": "prediccion", "version": version_modelo,
                         "prob": round(p, 4), "plan": cliente.plan, "ms": ms}))
    return Prediccion(probabilidad_abandono=round(p, 4), accion_retencion=p >= UMBRAL,
                      version_modelo=version_modelo)
""")

In [ ]:
import os
os.environ["MODELO_PATH"] = str(BASE / "artefactos" / "modelo.joblib")
from fastapi.testclient import TestClient
import tiendasur_churn.api as api
importlib.reload(api)

cliente_ok = {"antiguedad_meses": 3, "pedidos_ultimo_ano": 1, "ticket_medio": 25.0,
              "dias_desde_ultimo_pedido": 200, "incidencias_soporte": 3,
              "plan": "basico", "canal": "web"}

with TestClient(api.app) as http:          # el "with" ejecuta el arranque (carga del modelo)
    print("GET  /health  →", http.get("/health").json())
    r = http.post("/predict", json=cliente_ok)
    print("POST /predict →", r.status_code, r.json())
    malo = {**cliente_ok, "plan": "oro", "antiguedad_meses": -2}
    r = http.post("/predict", json=malo)
    print("POST /predict (datos inválidos) →", r.status_code)
    for err in r.json()["detail"]:
        print("   ", err["loc"][-1], "→", err["msg"])

## 8.4 · Tests: el contrato que protege tu trabajo

Tres tipos de test, todos rápidos:
- **De datos:** el esquema rechaza lo que debe rechazar.
- **De modelo:** calidad mínima (AUC ≥ 0,75 acordado con negocio), reproducibilidad y tests de **comportamiento** (más incidencias no puede bajar el riesgo).
- **De API:** respuestas correctas y errores `422` para entradas inválidas.

Fíjate en `conftest.py`: los tests **entrenan su propio modelo** en una carpeta temporal. En el CI no hay artefactos previos, y los tests no deben depender de nada que no creen ellos.

In [ ]:
escribir("tests/conftest.py", """
import pytest

from tiendasur_churn.datos import generar_datos
from tiendasur_churn.modelo import entrenar, guardar


@pytest.fixture(scope="session")
def df():
    return generar_datos(n=3000, semilla=7)


@pytest.fixture(scope="session")
def ruta_modelo(df, tmp_path_factory):
    \"\"\"Los tests entrenan su propio modelo: en CI no hay artefactos previos.\"\"\"
    pipe, metricas = entrenar(df)
    return guardar(pipe, metricas, df, tmp_path_factory.mktemp("artefactos"))
""")

escribir("tests/test_datos.py", """
import pytest

from tiendasur_churn.datos import validar_esquema


def test_esquema_valido(df):
    assert validar_esquema(df) is df


def test_rechaza_negativos(df):
    roto = df.copy()
    roto.loc[0, "ticket_medio"] = -5
    with pytest.raises(ValueError, match="negativos"):
        validar_esquema(roto)


def test_rechaza_columna_faltante(df):
    with pytest.raises(ValueError, match="faltan columnas"):
        validar_esquema(df.drop(columns=["plan"]))
""")

escribir("tests/test_modelo.py", """
from tiendasur_churn.modelo import entrenar


def test_supera_umbral_minimo_de_calidad(df):
    # Umbral acordado con negocio: si un cambio lo baja, el CI falla y no se despliega.
    _, metricas = entrenar(df)
    assert metricas["auc_test"] >= 0.75


def test_reproducible(df):
    _, m1 = entrenar(df, semilla=0)
    _, m2 = entrenar(df, semilla=0)
    assert m1 == m2


def test_mas_incidencias_no_baja_el_riesgo(df):
    # Test de comportamiento (direccional): lo que sabemos del negocio, escrito como test.
    pipe, _ = entrenar(df)
    base = df.drop(columns=["abandona", "cliente_id"]).iloc[[0]].copy()
    base["incidencias_soporte"] = 0
    peor = base.assign(incidencias_soporte=5)
    assert pipe.predict_proba(peor)[0, 1] >= pipe.predict_proba(base)[0, 1]
""")

escribir("tests/test_api.py", """
import pytest
from fastapi.testclient import TestClient

CLIENTE = {"antiguedad_meses": 3, "pedidos_ultimo_ano": 1, "ticket_medio": 25.0,
           "dias_desde_ultimo_pedido": 200, "incidencias_soporte": 3,
           "plan": "basico", "canal": "web"}


@pytest.fixture()
def http(ruta_modelo, monkeypatch):
    monkeypatch.setenv("MODELO_PATH", str(ruta_modelo))
    from tiendasur_churn.api import app
    with TestClient(app) as c:
        yield c


def test_health(http):
    assert http.get("/health").json()["estado"] == "ok"


def test_predict_devuelve_probabilidad(http):
    r = http.post("/predict", json=CLIENTE)
    assert r.status_code == 200
    assert 0 <= r.json()["probabilidad_abandono"] <= 1


@pytest.mark.parametrize("campo,valor", [("plan", "oro"), ("antiguedad_meses", -1),
                                         ("ticket_medio", 0)])
def test_rechaza_entradas_invalidas(http, campo, valor):
    assert http.post("/predict", json={**CLIENTE, campo: valor}).status_code == 422
""")

codigo_salida = ejecutar(sys.executable, "-m", "pytest", "-q")
print("\n→ tests", "OK" if codigo_salida == 0 else "FALLAN")

### Lint

`ruff` revisa estilo y errores comunes (imports sin usar, variables no definidas) en milisegundos. En el CI, si el lint falla, no se fusiona.

In [ ]:
codigo_ruff = ejecutar(sys.executable, "-m", "ruff", "check", ".")
print("→ ruff", "OK" if codigo_ruff == 0 else "con avisos (lee el mensaje: suelen ser imports sin usar o desordenados)")

## 8.5 · Docker: «en mi máquina funciona» ya no vale

Una imagen Docker empaqueta el código, el modelo **y** las versiones exactas de las librerías. Detalles que importan:
- las dependencias se copian **antes** que el código para aprovechar la caché de capas;
- `requirements-api.txt` fija las **mismas versiones** con las que se entrenó el modelo;
- la aplicación no corre como `root`;
- `.dockerignore` deja fuera tests, entornos y, sobre todo, **secretos** (`.env`).

In [ ]:
from importlib.metadata import version

# Mismas versiones que en el entrenamiento: un modelo joblib solo es fiable con la versión
# de scikit-learn con la que se guardó.
paquetes = ["scikit-learn", "pandas", "numpy", "joblib", "fastapi", "uvicorn", "pydantic"]
escribir("requirements-api.txt", "\n".join(f"{p}=={version(p)}" for p in paquetes) + "\n")

escribir("Dockerfile", """
FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1 \\
    PYTHONPATH=/app/src MODELO_PATH=/app/artefactos/modelo.joblib
WORKDIR /app

# 1) Dependencias primero: esta capa se cachea y no se reinstala si solo cambia el código.
COPY requirements-api.txt .
RUN pip install --no-cache-dir -r requirements-api.txt

# 2) Código y modelo.
COPY src/ src/
COPY artefactos/ artefactos/

# 3) Nunca como root.
RUN useradd --create-home app
USER app

EXPOSE 8000
HEALTHCHECK CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"
CMD ["uvicorn", "tiendasur_churn.api:app", "--host", "0.0.0.0", "--port", "8000"]
""")

escribir(".dockerignore", """
.git
.venv
__pycache__
tests
mlruns
*.ipynb
.env
""")

# Si tienes Docker instalado, pon esto a True para construir y probar la imagen.
CONSTRUIR_DOCKER = False
if CONSTRUIR_DOCKER and shutil.which("docker"):
    ejecutar("docker", "build", "-t", "tiendasur-churn:dev", ".")
else:
    print("\nPara construir y arrancar la imagen (en tu terminal, dentro de", BASE.name + "/):")
    print("  docker build -t tiendasur-churn:dev .")
    print("  docker run --rm -p 8000:8000 tiendasur-churn:dev")
    print("  curl localhost:8000/health   ·   documentación interactiva: http://localhost:8000/docs")

## 8.6 · CI/CD con GitHub Actions

El flujo de integración continua se ejecuta en cada *push* o *pull request*: **lint → tests → entrenamiento → imagen**. El despliegue (CD) añadiría un último paso: publicar la imagen en un registro (GHCR, ECR, Artifact Registry) y actualizar el servicio en la nube (Cloud Run, ECS, Azure Container Apps…).

In [ ]:
escribir(".github/workflows/ci.yml", """
name: ci
on:
  push:
    branches: [main]
  pull_request:

jobs:
  test-y-build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
          cache: pip
      - name: Dependencias
        run: pip install -r requirements-api.txt pytest httpx ruff
      - name: Lint
        run: ruff check .
      - name: Tests (incluye el umbral mínimo de AUC)
        run: pytest -q
      - name: Entrenar el artefacto
        run: PYTHONPATH=src python -m tiendasur_churn.entrenar --salida artefactos
      - name: Construir la imagen
        run: docker build -t tiendasur-churn:${{ github.sha }} .
""")
print("Cada push o pull request: lint → tests → entrenamiento → imagen. Si algo falla, no se fusiona.")

## 8.7 · Seguimiento de experimentos con MLflow

Cuando pruebas configuraciones, apuntarlas en una libreta no escala. MLflow registra parámetros, métricas, etiquetas (como la huella de los datos) y artefactos de cada ejecución, y permite compararlas y promover un modelo a producción con un **registro de modelos**.

In [ ]:
try:
    import mlflow
except ImportError:
    mlflow = None
    print("MLflow no está instalado (pip install mlflow). Saltamos esta sección.")

if mlflow:
    import logging
    logging.getLogger("mlflow").setLevel(logging.ERROR)
    mlflow.set_tracking_uri(f"sqlite:///{BASE / 'mlflow.db'}")   # local; en un equipo sería un servidor
    mlflow.set_experiment("tiendasur-churn")
    configuraciones = [
        {"max_depth": 3, "learning_rate": 0.05},
        {"max_depth": 6, "learning_rate": 0.1},
        {"max_depth": None, "learning_rate": 0.3},   # sin límite de profundidad y rápido: ¿sobreajusta?
    ]
    for params in configuraciones:
        with mlflow.start_run():
            _, m = modelo.entrenar(df, **params)
            mlflow.log_params({k: str(v) for k, v in params.items()})
            mlflow.log_metric("auc_test", m["auc_test"])
            mlflow.set_tag("huella_datos", modelo.huella_datos(df))
    runs = mlflow.search_runs(order_by=["metrics.auc_test DESC"])
    display(runs[["params.max_depth", "params.learning_rate", "metrics.auc_test", "tags.huella_datos"]])
    print("Para verlo en el navegador:  mlflow ui --backend-store-uri", f"sqlite:///{BASE / 'mlflow.db'}")

## 8.8 · Monitorización y deriva

Un modelo desplegado se degrada **sin dar errores**: el mundo cambia y los datos de producción dejan de parecerse a los de entrenamiento. Se vigilan tres cosas:
1. **Salud del servicio:** latencia, errores, peticiones por minuto (los logs JSON de la API).
2. **Deriva de datos:** ¿se parecen las entradas de hoy a las del entrenamiento? El **PSI** (*Population Stability Index*) lo resume por variable.
3. **Calidad real:** cuando llega la etiqueta (el cliente se fue o no), se recalcula la métrica. Suele llegar con semanas de retraso, por eso los puntos 1 y 2 son la alarma temprana.

In [ ]:
def psi(referencia, actual, bins: int = 10) -> float:
    """Population Stability Index. < 0,1 estable · 0,1–0,2 vigilar · > 0,2 deriva relevante."""
    cortes = np.unique(np.quantile(referencia, np.linspace(0, 1, bins + 1)))
    ref = np.histogram(referencia, cortes)[0] / len(referencia)
    act = np.histogram(np.clip(actual, cortes[0], cortes[-1]), cortes)[0] / len(actual)
    ref, act = np.clip(ref, 1e-6, None), np.clip(act, 1e-6, None)
    return float(np.sum((act - ref) * np.log(act / ref)))

# "Producción": un mes después. Subimos precios y empeoró el soporte.
produccion = datos.generar_datos(n=2000, semilla=99)
produccion["ticket_medio"] *= 1.35
produccion["incidencias_soporte"] += np.random.default_rng(1).poisson(0.8, len(produccion))

informe = pd.DataFrame({
    "psi": {c: psi(df[c], produccion[c]) for c in datos.NUMERICAS},
}).round(3)
informe["estado"] = pd.cut(informe["psi"], [-np.inf, 0.1, 0.2, np.inf],
                           labels=["estable", "vigilar", "DERIVA"])
display(informe)

p_ref = pipe.predict_proba(df[datos.NUMERICAS + datos.CATEGORICAS])[:, 1]
p_prod = pipe.predict_proba(produccion[datos.NUMERICAS + datos.CATEGORICAS])[:, 1]
print(f"PSI de las predicciones: {psi(p_ref, p_prod):.3f}  ·  % marcados para retención: "
      f"{(p_ref >= 0.5).mean():.0%} → {(p_prod >= 0.5).mean():.0%}")

## 8.9 · Trabajar con agentes de código

Hoy muchas ofertas piden soltura con asistentes como **Claude Code** o **Cursor**. Funcionan mucho mejor en un proyecto como este que en un notebook suelto, por tres motivos:
- **Los tests son la especificación:** le pides al agente un cambio y sabes si está bien porque `pytest` pasa (o no).
- **Un archivo de instrucciones** (`AGENTS.md` o `CLAUDE.md`) le dice cómo se ejecuta todo y qué no debe tocar.
- **Primero la intención, después el código:** escribe qué quieres, las restricciones y los criterios de aceptación (*spec-driven development*) antes de dejar que el agente programe. Y revisa el diff como revisarías el de un compañero: la responsabilidad sigue siendo tuya.

In [ ]:
escribir("AGENTS.md", """
# tiendasur-churn — instrucciones para agentes de código (Claude Code, Cursor, Codex…)

## Qué es
API que predice la probabilidad de abandono de un cliente de TiendaSur.

## Comandos
- Tests: `pytest -q` (deben pasar antes de cualquier commit)
- Lint: `ruff check .`
- Entrenar: `PYTHONPATH=src python -m tiendasur_churn.entrenar --salida artefactos`
- API local: `PYTHONPATH=src uvicorn tiendasur_churn.api:app --reload`

## Reglas
- Todo preprocesado va dentro del Pipeline de `modelo.py` (nunca fuera: fuga de datos).
- Cualquier campo nuevo de entrada: añádelo a `datos.py`, al esquema `Cliente` de `api.py` y a un test.
- No bajes el umbral de AUC de `tests/test_modelo.py` para que pase un test.
- No registres datos personales en los logs.
- No cambies versiones de `requirements-api.txt` sin reentrenar el modelo.
""")

## Ejercicios

Trabaja sobre los archivos del proyecto (puedes editarlos con `escribir(...)` o en tu editor) y vuelve a ejecutar `pytest` tras cada cambio.

1. **Predicción por lotes.** Añade a la API un endpoint `POST /predict/lote` que reciba una lista de clientes (máximo 500) y devuelva una lista de predicciones. Escribe un test que compruebe que el resultado tiene la misma longitud que la entrada y otro que compruebe que 501 clientes devuelven un 422.
2. **Contrato más estricto.** Un cliente no puede tener más pedidos en el último año que días transcurridos de antigüedad × 2 (regla de negocio inventada). Implementa la validación en el esquema `Cliente` con un `model_validator` de Pydantic y añade el test.
3. **¿Otro modelo?** Registra en MLflow una cuarta ejecución con `LogisticRegression` en lugar de `HistGradientBoostingClassifier` (manteniendo el mismo preprocesado). ¿Cuál desplegarías? Justifícalo en 3 líneas con AUC **y** un criterio no técnico (explicabilidad, latencia, coste de mantenimiento).
4. **Alarma de deriva.** Escribe una función `revisar_deriva(df_ref, df_prod) -> dict` que devuelva las variables con PSI > 0,2 y una recomendación (`"ok"`, `"vigilar"` o `"reentrenar"`). Decide tú la regla y explícala.
5. **Tu propia ficha de modelo.** Amplía `ficha.json` con: umbral de decisión, fecha de los datos, responsable y limitaciones conocidas (por ejemplo, «no validado con clientes de empresa»). ¿Por qué es útil en una auditoría, por ejemplo con el Reglamento europeo de IA?


In [ ]:
# Ejercicio 1 · tu código aquí


In [ ]:
# Ejercicio 2 · tu código aquí


In [ ]:
# Ejercicio 3 · tu código aquí


In [ ]:
# Ejercicio 4 · tu código aquí


In [ ]:
# Ejercicio 5 · tu código aquí
